# Analyse des agents IA :

## Lien utile

https://hyperskill.org/courses/149-ai-agents-theory-and-practice

Dans les systèmes d’agents utilisant des LLM, la gestion du raisonnement et de la collaboration entre agents est essentielle.
Les architectures modernes reposent sur plusieurs concepts clés :

- **Chain-of-Thought reasoning**
- **Orchestration patterns**
- **Multi-agent systems**
- **Delegation patterns**
- **Memory management**

Ces concepts permettent de construire des systèmes d’agents capables de résoudre des problèmes complexes de manière structurée, fiable et explicable.

## Chain-of-Thought (CoT) ( on l a directement insere cette etape dans le systeme prompt !! )

Le **Chain-of-Thought** est une technique de raisonnement qui consiste à demander explicitement au modèle de réfléchir étape par étape avant de produire une réponse.

La méthode la plus simple consiste à ajouter dans le prompt la phrase :

"Réfléchis étape par étape"

Cela pousse le modèle à produire une **suite logique de raisonnements** avant de donner la réponse finale.

### Exemple simple

Sans Chain-of-Thought :

Question → Réponse directe

Avec Chain-of-Thought :

Question → Raisonnement intermédiaire → Conclusion

### Implémentation dans les agents

Dans les architectures d’agents, on peut implémenter le CoT de plusieurs façons :

1. Ajouter **"think step by step"** dans le prompt système
2. Forcer une **structure de raisonnement**
3. Utiliser des **sorties structurées (JSON schema)**

Les API modernes comme OpenAI permettent d'imposer une structure de raisonnement avec des **schemas JSON**, ce qui garantit que le modèle suit un schéma logique cohérent.

### Avantage principal

Le Chain-of-Thought améliore :

- la **transparence du raisonnement**
- la **débogabilité**
- la **confiance utilisateur**

On peut voir comment l'agent arrive à sa conclusion.

In [4]:
import ollama

# Illustration du Chain-of-Thought avec Ollama et le modèle Mistral
prompt = """
Résous le problème suivant.

Question : Si un train roule à 100 km/h pendant 2 heures, quelle distance parcourt-il ?

Réfléchis étape par étape.
"""

response = ollama.chat(
    model="mistral",
    messages=[{"role": "user", "content": prompt}]
)

print(response['message']['content'])

RemoteProtocolError: Server disconnected without sending a response.

## Orchestration Patterns dans les systèmes multi-agents ( on a utilise un code-based architecture avec maestro !! )

Les **patterns d'orchestration** définissent comment plusieurs agents collaborent pour résoudre un problème. Ils déterminent :

- comment les agents communiquent
- qui contrôle la décision
- comment les tâches sont distribuées

### Points importants

**1. Localisation du contrôle**

Deux architectures possibles :

Centralisée :
Un agent coordinateur dirige tout le système.

Décentralisée :
Les agents prennent des décisions localement.

Centralisé → plus simple à contrôler
Décentralisé → plus flexible et robuste

---

**2. LLM vs Code**

LLM-based decisions :

+ flexible
+ adaptatif

Mais :

- moins prévisible

Code-based logic :

+ plus déterministe
+ plus stable

Mais :

- moins flexible

---

**3. Adapter le pattern au problème**

Pipeline → tâches séquentielles
Group chat → problèmes ouverts
Debate → problèmes nécessitant une forte précision

---

**4. Composition des patterns**

Les systèmes avancés combinent plusieurs patterns.

Par exemple :

Router agent → pipeline → débat final.

---

**5. Observabilité**

Les systèmes multi-agents sont des systèmes distribués.

Il est donc crucial de monitorer :

- les décisions des agents
- les échanges
- les erreurs

---

**6. Commencer simple**

La plupart des problèmes ne nécessitent pas une architecture multi-agent complexe.

Commencer avec un agent unique puis complexifier si nécessaire.

## Multi-Agent Systems ( dans notre cas on a un agent unique ( on peut considerer les differents outils comme des mini-agents ) !! )

Les systèmes multi-agents représentent une évolution importante dans les architectures basées sur LLM.

Au lieu d’un **agent unique très complexe**, on construit une **équipe d’agents spécialisés**.

### Avantages

**Spécialisation**

Chaque agent se concentre sur une tâche précise.

Exemples :

- agent recherche
- agent analyse
- agent rédaction

---

**Efficacité du contexte**

Chaque agent reçoit uniquement les informations nécessaires.

Cela réduit :

- le coût
- la latence
- le bruit informationnel

---

**Scalabilité**

On peut lancer plusieurs instances d'un agent très utilisé.

Exemple :

10 agents de recherche
1 agent de synthèse

---

**Isolation**

On peut modifier un agent sans impacter les autres.

---

### Limites

Les systèmes multi-agents introduisent :

- de la complexité
- des problèmes de coordination
- des besoins en mémoire partagée

Pour des tâches simples, un agent unique reste préférable.

## Pattern de délégation ( un peu pres ce qu on fait avec maestro !! )

Le pattern de **délégation** imite la structure d'une organisation humaine.

Un agent coordinateur reçoit une tâche complexe et la divise en sous-tâches confiées à des agents spécialisés.

### Fonctionnement

1. Le coordinateur reçoit la requête utilisateur
2. Il décompose la tâche
3. Il délègue les sous-tâches aux agents experts
4. Il combine les résultats

### Exemple

Tâche : créer un rapport de marché

Coordination :

Recherche → agent de recherche
Analyse → agent d'analyse
Rédaction → agent de rédaction

Le coordinateur assemble ensuite le rapport final.

### Avantages

- meilleure qualité
- agents spécialisés
- architecture modulaire

In [3]:
# Définition simple d'un agent et d'un coordinateur pour illustrer le pattern de délégation
class Agent:
    def __init__(self, nom, role):
        self.nom = nom
        self.role = role

    def executer(self, tache):
        print(f"{self.nom} traite la tâche : {tache}")


agent_recherche = Agent("Agent Recherche", "collecte d'informations")
agent_analyse = Agent("Agent Analyse", "analyse des données")
agent_redaction = Agent("Agent Rédaction", "rédaction du rapport")


def coordinateur(tache):
    agent_recherche.executer("chercher des informations")
    agent_analyse.executer("analyser les données")
    agent_redaction.executer("rédiger le rapport final")


coordinateur("rapport de marché")

Agent Recherche traite la tâche : chercher des informations
Agent Analyse traite la tâche : analyser les données
Agent Rédaction traite la tâche : rédiger le rapport final


## Types de mémoire pour les agents

Les agents IA utilisent plusieurs types de stockage pour gérer l'information.

### 1. Mémoire en mémoire vive (In-Memory) ( dans notre cas c est l a classe implemente dans le fihier memoire courte !! )

Utilisée pour :

- l'état de session
- la conversation active

Très rapide mais temporaire.

---

### 2. Bases vectorielles ( literalement ce qu on a mis dans le code de memoire-RAG en vectorisant la knowledge base !! )

Stockent des **embeddings sémantiques**.

Utilisées pour :

- recherche sémantique
- RAG (Retrieval Augmented Generation)

Exemples :

- FAISS
- Chroma ( on a utilise cette lib pour sa simple utilisation !! )
- Pinecone

---

### 3. Document Stores ( on a utilise SQLite pour stocker les tiketss a analyser apres !! !! )

Stockent des informations structurées.

Utiles pour :

- requêtes structurées
- extraction de faits

Exemples :

- MongoDB
- Elasticsearch

---

### 4. Graph databases

Stockent des **relations entre entités**.

Utiles pour :

- knowledge graphs
- relations complexes

Exemples :

- Neo4j

## Défis de la mémoire long terme

La gestion de la mémoire long terme pose plusieurs questions :

### Que stocker ?

Toutes les informations ne méritent pas d'être mémorisées.

### Comment stocker ?

Options :

- texte brut
- embeddings
- données structurées

### Comment récupérer l'information ?

- recherche par mots-clés
- recherche sémantique
- recherche temporelle

### Quand récupérer la mémoire ?

- à chaque requête
- seulement quand nécessaire

### Comment mettre à jour ?

- écraser
- ajouter une nouvelle version
- versioning

### Quand oublier ?

- expiration
- score de pertinence
- mécanismes de decay

## Gestion et maintenance de la mémoire ( on a pas encore fait cela, on utilise la mtd stupide de garde tous !! )


La mémoire d'un agent doit être activement maintenue.

Sans cela, elle devient :

- trop volumineuse
- contradictoire
- lente à interroger

### Mise à jour

Ajouter les nouvelles informations avec timestamp est souvent la stratégie la plus simple.

### Décroissance de mémoire

Techniques courantes :

TTL (Time To Live)

Les données expirent après un certain temps.

LRU (Least Recently Used)

Supprime les données les moins récemment utilisées.

LFU (Least Frequently Used)

Supprime les données les moins utilisées.

---

### Bonnes pratiques

- attribuer des durées de vie différentes selon les types de mémoire
- marquer certaines mémoires comme permanentes
- nettoyer automatiquement les anciennes données
- surveiller la croissance du stockage

La mémoire d’un agent est comme un jardin : elle doit être régulièrement entretenue pour rester utile et efficace.

## Importance du contexte

Dans les LLM :

Tout doit être explicite.

Si une information n'est pas dans le **contexte du prompt**, le modèle ne peut pas la connaître.

### Impact du contexte

Contexte plus grand :

+ plus d'information disponible

Mais :

- coût plus élevé
- latence plus importante

### Qualité du contexte

Un contexte mal sélectionné peut dégrader les performances.

Un bon système d'agents doit donc :

- sélectionner les informations pertinentes
- éviter le bruit informationnel
- optimiser la taille du contexte.

## Orchestration de systèmes multi-agents

Les systèmes d'IA modernes ne reposent plus uniquement sur un seul modèle.
On utilise souvent **plusieurs agents spécialisés** qui collaborent pour résoudre une tâche complexe.

Chaque agent possède :

- un rôle spécifique
- des outils particuliers
- des instructions dédiées

Un **orchestrateur** est responsable de coordonner ces agents.

Plusieurs architectures d'orchestration existent :

1. Système avec coordinateur central
2. Pipeline séquentiel
3. Orchestrateur dynamique
4. Conversation multi-agents
5. Système avec transfert de tâche (handoff)

Ces architectures sont très utilisées dans les systèmes de **recherche automatisée, assistants complexes et agents autonomes**.

### Système avec coordinateur central ( je suis pas sur , mais utiliser des outils ressemble un peu a ca :) !! )

Dans cette architecture, un **agent coordinateur** décompose une tâche complexe en sous-tâches et les délègue à des agents spécialisés.

Chaque agent a une compétence particulière :

- recherche web
- analyse de données
- rédaction

Le coordinateur collecte ensuite les résultats et produit la réponse finale.

In [6]:
# Architecture avec un agent coordinateur déléguant des tâches à des experts spécialisés
class SystemeRecherche:

    def __init__(self):

        self.coordinateur = Agent(
            instructions="Décomposer les tâches de recherche et déléguer",
            handoffs=[self.chercheur_web, self.analyste_donnees, self.redacteur]
        )

        self.chercheur_web = Agent(
            instructions="Trouver et résumer des articles pertinents",
            tools=[recherche_web, extraire_contenu]
        )

        self.analyste_donnees = Agent(
            instructions="Analyser les données quantitatives et les tendances",
            tools=[interroger_base, calculer_metriques]
        )

        self.redacteur = Agent(
            instructions="Synthétiser les résultats dans un rapport cohérent",
            tools=[creer_document]
        )

### Orchestration séquentielle ( je pense que ca erait utile pour nous de l ajouter au futur , car cela permet de mettre differents par exemple agents de la chaine SAV ( savoir le probleme , resoudre le robleme , envoyer un technicien , recevoir un rapport ... ) !! )

Dans ce modèle, les agents travaillent **les uns après les autres**.

Chaque agent reçoit le résultat produit par le précédent.

Ce type d'architecture est particulièrement utile pour les **pipelines de génération de contenu**.

In [7]:
# Pipeline où chaque agent traite le résultat de l'agent précédent de manière séquentielle
class OrchestrateurSequentiel:

    def __init__(self, specialistes):
        self.specialistes = specialistes

    def executer(self, tache):

        resultat = tache

        for specialiste in self.specialistes:
            resultat = specialiste.run(resultat)

        return resultat

pipeline = OrchestrateurSequentiel([
    agent_recherche,
    agent_plan,
    agent_redaction,
    agent_edition,
    agent_seo
])

article_blog = pipeline.executer("Écrire un article sur l'IA dans la santé")

NameError: name 'agent_plan' is not defined

### Orchestration dynamique ( le maestro implemente ca dans le contexte ou c est lui qui decide a l etape de THOUGHT qu elle outil utiliser !! )

Dans ce modèle, un **agent manager** décide dynamiquement :

- quelles sous-tâches doivent être exécutées
- quel agent est le plus adapté pour chaque tâche
- comment adapter le plan en fonction des résultats obtenus

Ce type d'orchestration permet une **plus grande flexibilité**.

In [8]:
# Un manager planifie et sélectionne dynamiquement l'agent approprié en fonction de l'avancement
class OrchestrateurDynamique:

    def __init__(self, manager, pool_specialistes):
        self.manager = manager
        self.specialistes = pool_specialistes

    def executer(self, tache):

        plan = self.manager.creer_plan(tache)
        resultats = {}

        while not plan.est_termine():

            sous_tache = plan.prochaine_tache()

            specialiste = self.manager.selectionner_specialiste(sous_tache)

            resultat = specialiste.run(sous_tache)

            resultats[sous_tache.id] = resultat

            plan = self.manager.mettre_a_jour_plan(plan, resultat)

        return self.manager.synthetiser(resultats)

### Conversation multi-agents ( un peu le truc de OPENCLAW , ca serait util de laisser des chatbot SAV communiquer apres vente , mais j ai un mauvais presentiment sur cela ( car ca va conduir a ce que les chatbots influencent d autres ce qui va aboutir a leur halluvcination ) !! )

Dans certaines architectures, plusieurs agents participent à une **discussion collaborative** pour résoudre un problème.

Chaque agent peut :

- proposer des idées
- critiquer les propositions
- améliorer les réponses

Le système sélectionne l'agent qui doit parler à chaque tour de conversation.

In [9]:
# Collaboration entre agents via une conversation où chaque intervenant est sélectionné par un LLM
class OrchestrateurConversation:

    def __init__(self, participants, max_tours=20):

        self.participants = participants
        self.conversation = []
        self.max_tours = max_tours

    def selectionner_prochain_intervenant(self):

        prompt_selection = """
        Étant donné la conversation actuelle,
        quel agent devrait intervenir pour faire avancer la discussion ?
        """

        return self.llm_selection(self.conversation, self.participants)

    def executer(self, tache):

        self.conversation.append({"role": "utilisateur", "contenu": tache})

        for tour in range(self.max_tours):

            intervenant = self.selectionner_prochain_intervenant()

            message = intervenant.generer(self.conversation)

            self.conversation.append({
                "role": intervenant.nom,
                "contenu": message
            })

            if self.conclusion_atteinte():
                break

        return self.synthetiser_resultat()

### Architecture avec transfert de tâche (Handoff) ( utile dans le futur ( si on a le temps apres ) car ca permet de dimunier la taille de chaque chatbot et reduire les cout de dev. !! )

Dans cette architecture, un agent peut **transférer une tâche à un autre agent** lorsqu'il estime que celui-ci est plus compétent.

Cela permet de créer un **réseau d'agents spécialisés**.

Ce type d'architecture est souvent utilisé dans les systèmes de support technique ou d'assistance automatisée.

In [10]:
# Passage de relais (transfert) entre agents spécialisés jusqu'à la résolution de la tâche
class OrchestrateurHandoff:

    def __init__(self, agent_entree, reseau_agents):

        self.agent_entree = agent_entree
        self.reseau = reseau_agents

    def executer(self, tache):

        agent_actuel = self.agent_entree

        conversation = [
            {"role": "utilisateur", "contenu": tache}
        ]

        for _ in range(10):

            resultat = agent_actuel.process(conversation)

            if resultat.est_termine:
                return resultat.sortie

            if resultat.transfert_vers:

                agent_suivant = self.reseau.get(resultat.transfert_vers)

                conversation.append({
                    "role": agent_actuel.nom,
                    "contenu": f"Transfert : {resultat.contexte}"
                })

                agent_actuel = agent_suivant

### Comparaison des architectures multi-agents

| Architecture | Avantages | Limites |
|--------------|-----------|---------|
| Coordinateur central | Simple à contrôler | Peut devenir un goulot d'étranglement |
| Pipeline séquentiel | Très structuré | Peu flexible |
| Orchestration dynamique | Adaptatif | Plus complexe |
| Conversation multi-agents | Collaboration riche | Coût computationnel élevé |
| Handoff entre agents | Très modulaire | Coordination plus difficile |

Ces architectures sont au cœur des **systèmes d'agents modernes** et permettent de résoudre des tâches complexes de manière collaborative.

## Gestion de la mémoire à court terme dans les agents IA

On explore ICI les différentes stratégies pour gérer la **mémoire à court terme dans les agents conversationnels basés sur des LLM**.

Les modèles de langage ont une **fenêtre de contexte limitée**, ce qui signifie qu'ils ne peuvent pas conserver une conversation infinie.

Il est donc nécessaire d'utiliser des stratégies permettant de gérer efficacement l'historique de conversation :

1. Conserver tous les messages
2. Fenêtre glissante
3. Filtrage basé sur l'importance
4. Résumé des anciens messages

Ces stratégies sont couramment utilisées dans les **architectures d'agents conversationnels modernes**.

### Structure de l'historique de conversation

Nous représentons la conversation comme une **liste de messages**.

Chaque message est un dictionnaire contenant deux champs :

- **role** : l'auteur du message (`utilisateur`, `assistant`, `systeme`)
- **contenu** : le texte du message

Exemple :

```python
{
    "role": "utilisateur",
    "contenu": "Je n'arrive pas à me connecter au VPN"
}

In [ ]:
# Classe gérant une fenêtre glissante limitée d'historique de conversation
class MemoireCourtTerme:

    def __init__(self, nombre_max_messages=50):
        self.historique_conversation = []
        self.nombre_max_messages = nombre_max_messages

    def ajouter_message(self, role, contenu):

        self.historique_conversation.append({
            "role": role,
            "contenu": contenu
        })

        # Suppression des messages les plus anciens si la limite est dépassée
        if len(self.historique_conversation) > self.nombre_max_messages:
            self.historique_conversation = self.historique_conversation[-self.nombre_max_messages:]

    def obtenir_historique(self):
        return self.historique_conversation

### Stratégie 1 : Conserver tous les messages ( ce qu on fesait au debut !! )

La stratégie la plus simple consiste à **conserver l'intégralité de l'historique de conversation**.

Avantages :

- Contexte maximal pour le modèle
- Aucune perte d'information

Inconvénients :

- La taille du contexte augmente rapidement
- Coût en tokens plus élevé
- Peut dépasser la fenêtre de contexte du modèle

Cette approche est rarement utilisée dans les systèmes réels lorsque les conversations sont longues.

In [11]:
def conserver_tout(historique_conversation, nouveau_message):
    """Approche simple : conserver tous les messages"""

    historique_conversation.append(nouveau_message)

    return historique_conversation

### Stratégie 2 : Fenêtre glissante ( ce qu on fait maintenant , mais c est dangereux , a revoir , car il y a le risque que le LLM oublie completement la requete original de lutilisateur !! )

La stratégie de **fenêtre glissante** consiste à ne conserver que les **N derniers messages** de la conversation.

Lorsque la taille maximale est atteinte, les messages les plus anciens sont supprimés.

Avantages :

- Implémentation simple
- Utilisation mémoire contrôlée
- Le contexte récent est conservé

Inconvénients :

- Des informations importantes plus anciennes peuvent être perdues

In [12]:
def fenetre_glissante(historique_conversation, nouveau_message, taille_fenetre=10):
    """Conserver uniquement les N derniers messages"""

    historique_conversation.append(nouveau_message)

    if len(historique_conversation) > taille_fenetre:
        historique_conversation = historique_conversation[-taille_fenetre:]

    return historique_conversation

### Stratégie 3 : Filtrage basé sur l'importance ( ce qu on cherche a implementer ( c est complexe certes , mais c est le meilleur ) !! )

Au lieu de conserver tous les messages, on peut choisir de **ne garder que les messages jugés importants**.

L'importance peut être déterminée par :

- des règles simples
- un classificateur
- un modèle de langage
- des mots-clés

Dans cet exemple, nous utilisons une fonction simple basée sur des mots-clés.

In [13]:
def est_important(message):
    """Détection simple des messages importants"""

    mots_cles = ["erreur", "critique", "mot de passe", "vpn"]

    contenu = message["contenu"].lower()

    return any(mot in contenu for mot in mots_cles)


def filtrage_par_importance(historique_conversation, nouveau_message):
    """Ne conserver que les messages importants"""

    if est_important(nouveau_message):
        historique_conversation.append(nouveau_message)

    return historique_conversation

### Stratégie 4 : Résumé des anciens messages ( je pense qu on va implementer ca dans le futur , c est moins complexe que la mtd precedente , et on peut facilement deleguer la tache de resumer au LLM lui meme !! )

Lorsque l'historique devient trop long, il est possible de **résumer les anciens messages** afin de réduire la taille du contexte tout en conservant l'information essentielle.

Cette technique permet :

- de conserver le contexte global
- de réduire le nombre de tokens
- d'améliorer l'efficacité du modèle

Cette stratégie est souvent utilisée dans les **agents conversationnels avancés**.

In [14]:
def resumer_messages(messages):
    """Fonction simplifiée de résumé"""

    texte_combine = " ".join([m["contenu"] for m in messages])

    return {
        "role": "systeme",
        "contenu": f"Résumé de la conversation précédente : {texte_combine[:100]}..."
    }


def strategie_resume(historique_conversation, seuil=20):
    """Résumer les anciens messages lorsque l'historique devient trop long"""

    if len(historique_conversation) > seuil:

        resume = resumer_messages(historique_conversation[:seuil])

        historique_conversation = [resume] + historique_conversation[seuil:]

    return historique_conversation

## Mémoire à long terme dans les agents IA

Contrairement à la mémoire à court terme qui gère l'historique immédiat d'une conversation, la **mémoire à long terme** permet à un agent de conserver des informations sur une longue durée.

Elle permet notamment :

- de conserver des connaissances
- de mémoriser des interactions passées
- d'apprendre des procédures efficaces
- de personnaliser les interactions avec les utilisateurs

Dans les architectures modernes d'agents IA, la mémoire à long terme est souvent divisée en plusieurs catégories inspirées des sciences cognitives :

1. **Mémoire épisodique** : souvenirs d'événements ou d'interactions passées ( c est un peu dans la zone grise, doit t on garder les discussions avec les cliens pour ameliorer notre chatbot ou pas ?? )
2. **Mémoire procédurale** : procédures ou méthodes pour accomplir des tâches ( c est literallement la knowledge base ( dans ce cas michelin ) de notre projet !! )
3. **Mémoire sémantique** : connaissances factuelles ( cela serait dans ce cas ce que le modele utilise ( mistral .. ) s y ete entraine avant !! )
4. **Système hybride** combinant ces différentes formes de mémoire

### Structure générale d'une mémoire à long terme

Une mémoire à long terme utilise généralement un **backend de stockage** (base de données, vector store, fichier, etc.).

Elle permet :

- de stocker des informations
- de récupérer les informations pertinentes
- d'associer des métadonnées

Le code suivant montre une structure simplifiée de mémoire long terme.

In [30]:
class MemoireLongTerme:

    def __init__(self, stockage_backend):
        self.stockage = stockage_backend

    def stocker(self, cle, valeur, metadata=None):
        """Stocker une information avec des métadonnées optionnelles"""

        self.stockage.save(cle, {
            "valeur": valeur,
            "metadata": metadata,
            "timestamp": datetime.now()
        })

    def recuperer(self, requete):
        """Récupérer des souvenirs pertinents"""

        return self.stockage.search(requete)

### Mémoire épisodique

La **mémoire épisodique** correspond aux souvenirs d'interactions passées.

Dans le contexte d'un agent conversationnel, cela peut inclure :

- les conversations précédentes avec un utilisateur
- les événements importants
- les préférences de l'utilisateur

Cette mémoire permet à l'agent de **personnaliser ses réponses** et de se souvenir du contexte des interactions passées.

In [31]:
class MemoireEpisodique:

    def stocker_episode(self, id_utilisateur, episode):

        entree_memoire = {
            "id_utilisateur": id_utilisateur,
            "timestamp": datetime.now(),
            "id_conversation": str(uuid.uuid4()),
            "resume": self.resumer_episode(episode),
            "evenements_cles": self.extraire_evenements(episode),
            "preferences_utilisateur": self.extraire_preferences(episode)
        }

        self.base_donnees.insert("memoires_episodiques", entree_memoire)

    def rappeler_historique_utilisateur(self, id_utilisateur, contexte):

        """Récupérer les interactions passées pertinentes"""

        memoires = self.base_donnees.query(
            "memoires_episodiques",
            filter={"id_utilisateur": id_utilisateur},
            sort_by="pertinence_contexte",
            limit=5
        )

        return memoires

### Mémoire procédurale

La **mémoire procédurale** contient les **procédures permettant d'accomplir des tâches**.

Elle est particulièrement utile pour :

- automatiser certaines actions
- apprendre à partir des succès passés
- améliorer l'efficacité de l'agent

Exemple :
Un agent de support IT peut apprendre la meilleure procédure pour résoudre un problème VPN.

In [32]:
class MemoireProcedurale:

    def apprendre_procedure(self, type_tache, etapes_reussies, resultat):

        procedure = {
            "type_tache": type_tache,
            "etapes": etapes_reussies,
            "taux_succes": self.calculer_taux_succes(resultat),
            "optimisations": self.identifier_optimisations(etapes_reussies),
            "prerequis": self.extraire_prerequis(etapes_reussies)
        }

        self.base_donnees.upsert("procedures", procedure)

    def obtenir_procedure(self, type_tache):

        """Récupérer la meilleure procédure pour une tâche"""

        return self.base_donnees.find_one(
            "procedures",
            filter={"type_tache": type_tache},
            sort_by="taux_succes"
        )

### Mémoire sémantique

La **mémoire sémantique** contient des **connaissances factuelles**.

Dans les systèmes modernes d'IA, cette mémoire est souvent implémentée avec :

- une base vectorielle
- des embeddings
- une recherche sémantique

Cette approche est largement utilisée dans les systèmes **RAG (Retrieval Augmented Generation)**.

In [33]:
# Gestion de la mémoire à long terme sémantique utilisant des embeddings et une base vectorielle
class MemoireSemantique:

    def __init__(self):
        self.base_vecteurs = BaseVectorielle()

    def ajouter_connaissance(self, fait, categorie, metadata=None):

        embedding = self.generer_embedding(fait)

        self.base_vecteurs.insert({
            "contenu": fait,
            "embedding": embedding,
            "categorie": categorie,
            "metadata": metadata or {},
            "confiance": self.evaluer_confiance(fait)
        })

    def rechercher_connaissance(self, requete, categorie=None):

        embedding_requete = self.generer_embedding(requete)

        resultats = self.base_vecteurs.search(
            embedding_requete,
            filter={"categorie": categorie} if categorie else None,
            top_k=5
        )

        return resultats

### Système de mémoire hybride

Les agents avancés combinent souvent plusieurs types de mémoire :

- mémoire épisodique
- mémoire procédurale
- mémoire sémantique

Cette architecture permet à l'agent de :

- se souvenir des interactions passées
- apprendre de nouvelles procédures
- accéder à des connaissances générales

On parle alors de **système de mémoire hybride**.

In [34]:
# Coordination de plusieurs types de mémoires (épisodique, procédurale, sémantique) pour un agent complet
class SystemeMemoireHybride:

    def __init__(self):

        self.memoire_episodique = MemoireEpisodique()
        self.memoire_procedurale = MemoireProcedurale()
        self.memoire_semantique = MemoireSemantique()

    def traiter_interaction(self, id_utilisateur, conversation):

        # Stocker l'épisode
        self.memoire_episodique.stocker_episode(id_utilisateur, conversation)

        # Extraire les faits appris
        faits = self.extraire_faits(conversation)

        for fait in faits:
            self.memoire_semantique.ajouter_connaissance(fait)

        # Apprendre une procédure si une tâche a été complétée
        tache_terminee = self.identifier_tache_completee(conversation)

        if tache_terminee:

            self.memoire_procedurale.apprendre_procedure(
                tache_terminee.type,
                tache_terminee.etapes,
                tache_terminee.resultat
            )

    def preparer_contexte(self, id_utilisateur, requete_actuelle):

        """Récupérer les informations pertinentes pour répondre"""

        return {
            "historique_utilisateur": self.memoire_episodique.rappeler_historique_utilisateur(id_utilisateur),
            "procedures_pertinentes": self.memoire_procedurale.obtenir_procedure(requete_actuelle),
            "connaissances": self.memoire_semantique.rechercher_connaissance(requete_actuelle)
        }